In [2]:
import pandas as pd

# Reload (or reuse from previous notebook if same session)
transaction = pd.read_excel("../data/Transaction.xlsx")
user = pd.read_excel("../data/User.xlsx")
city = pd.read_excel("../data/City.xlsx")
continent = pd.read_excel("../data/Continent.xlsx")
country = pd.read_excel("../data/Country.xlsx")
region = pd.read_excel("../data/Region.xlsx")
attraction_type = pd.read_excel("../data/Type.xlsx")
visit_mode = pd.read_excel("../data/Mode.xlsx")
item = pd.read_excel("../data/Updated_Item.xlsx")

tables = {
    "Transaction": transaction, "User": user, "City": city,
    "Continent": continent, "Country": country, "Region": region,
    "Type": attraction_type, "Mode": visit_mode, "Item": item
}

# 1. Check missing values in every table
print("### MISSING VALUES ###")
for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing)
print("\n(if nothing printed above besides headers, no missing values found)\n")

# 2. Check duplicates
print("### DUPLICATE ROWS ###")
for name, df in tables.items():
    dupes = df.duplicated().sum()
    if dupes > 0:
        print(f"{name}: {dupes} duplicate rows")

# 3. Investigate the weird AttractionTypeId column in Item
print("\n### Item.AttractionTypeId unique values (why is this 'object' dtype?) ###")
print(item['AttractionTypeId'].apply(type).value_counts())
print(item['AttractionTypeId'].unique())

# 4. Check those placeholder "-" rows we spotted (id=0 rows)
print("\n### Placeholder rows (ID=0) ###")
print("Continent 0:", continent[continent['ContinentId']==0])
print("Country 0:", country[country['CountryId']==0])
print("Region 0:", region[region['RegionId']==0])
print("Mode 0:", visit_mode[visit_mode['VisitModeId']==0])
print("City 0:", city[city['CityId']==0])

# 5. Check ranges/outliers in Rating and VisitYear/Month
print("\n### Transaction.Rating value counts ###")
print(transaction['Rating'].value_counts().sort_index())
print("\n### VisitYear range ###", transaction['VisitYear'].min(), "-", transaction['VisitYear'].max())
print("### VisitMonth range ###", transaction['VisitMonth'].min(), "-", transaction['VisitMonth'].max())

# 6. Check if any UserId/AttractionId/VisitMode in Transaction don't exist in reference tables (orphan records)
print("\n### Orphan check ###")
print("Users in Transaction not in User table:", (~transaction['UserId'].isin(user['UserId'])).sum())
print("Attractions in Transaction not in Item table:", (~transaction['AttractionId'].isin(item['AttractionId'])).sum())
print("VisitModes in Transaction not in Mode table:", (~transaction['VisitMode'].isin(visit_mode['VisitModeId'])).sum())

### MISSING VALUES ###

User:
CityId    4
dtype: int64

City:
CityName    1
dtype: int64

(if nothing printed above besides headers, no missing values found)

### DUPLICATE ROWS ###

### Item.AttractionTypeId unique values (why is this 'object' dtype?) ###
AttractionTypeId
<class 'str'>    1668
<class 'int'>      30
Name: count, dtype: int64
[13 63 76 72 93 92 61 64 82 91 84 19 34 10 2 45 44 'Temple' 'Beach' 'Park'
 'Museum' 'Market']

### Placeholder rows (ID=0) ###
Continent 0:    ContinentId Continent
0            0         -
Country 0:    CountryId Country  RegionId
0          0       -         0
Region 0:   Region  RegionId  ContinentId
0      -         0            0
Mode 0:    VisitModeId VisitMode
0            0         -
City 0:    CityId CityName  CountryId
0       0        -          0

### Transaction.Rating value counts ###
Rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64

### VisitYear range ### 2013 - 2022
### VisitMonth range ### 1

# Understand the AttractionTypeId problem

In [3]:
# Let's look at a sample of the "bad" rows (where AttractionTypeId is text, not a number)
bad_rows = item[item['AttractionTypeId'].apply(lambda x: isinstance(x, str))]
print(bad_rows.head(10))

# What are ALL the unique text values in there?
print("\nUnique text values:")
print(bad_rows['AttractionTypeId'].unique())

    AttractionId  AttractionCityId AttractionTypeId  \
30          1298                 0           Temple   
31          1299                 0            Beach   
32          1300                 0             Park   
33          1301                 0           Museum   
34          1302                 2           Museum   
35          1303                 2           Market   
36          1304                 2           Market   
37          1305                 2           Temple   
38          1306                 1             Park   
39          1307                 1           Museum   

                        Attraction                    AttractionAddress  
30              Ancient Temple - -                    Main Street, -, -  
31              Paradise Beach - -                    Main Street, -, -  
32               National Park - -                    Main Street, -, -  
33    Cultural Heritage Center - -                    Main Street, -, -  
34      Art Gallery - So

# Fix the AttractionTypeId column

In [4]:
attraction_type = pd.read_excel("../data/Type.xlsx")
print(attraction_type)

    AttractionTypeId                  AttractionType
0                  2                   Ancient Ruins
1                 10                         Ballets
2                 13                         Beaches
3                 19                 Caverns & Caves
4                 34           Flea & Street Markets
5                 44                  Historic Sites
6                 45                 History Museums
7                 61                  National Parks
8                 63         Nature & Wildlife Areas
9                 64                   Neighborhoods
10                72  Points of Interest & Landmarks
11                76                 Religious Sites
12                82                            Spas
13                84              Speciality Museums
14                91                        Volcanos
15                92                     Water Parks
16                93                      Waterfalls


# Fix the AttractionTypeId column

In [5]:
# Build the mapping from text name -> correct numeric ID
type_mapping = {
    'Temple': 76,   # Religious Sites
    'Beach': 13,    # Beaches
    'Park': 61,     # National Parks
    'Market': 34,   # Flea & Street Markets
    'Museum': 45    # History Museums
}

# Replace text values with correct numeric IDs
item['AttractionTypeId'] = item['AttractionTypeId'].replace(type_mapping)

# Now convert the whole column to integer type
item['AttractionTypeId'] = item['AttractionTypeId'].astype(int)

# Verify the fix worked
print(item['AttractionTypeId'].dtype)          # should say int64 now
print(item['AttractionTypeId'].unique())        # should be all numbers, no text

int64
[13 63 76 72 93 92 61 64 82 91 84 19 34 10  2 45 44]


# Handle missing values

In [6]:
# Reload user and city if not already in this session
user = pd.read_excel("../data/User.xlsx")
city = pd.read_excel("../data/City.xlsx")

# Show the 4 User rows with missing CityId
print("Users with missing CityId:")
print(user[user['CityId'].isnull()])

# Show the 1 City row with missing CityName
print("\nCity with missing CityName:")
print(city[city['CityName'].isnull()])

Users with missing CityId:
       UserId  ContinentId  RegionId  CountryId  CityId
2279     7175            5        17        135     NaN
6027    17595            1         4         22     NaN
21303   56972            5        17        135     NaN
25494   67461            5        17        135     NaN

City with missing CityName:
      CityId CityName  CountryId
6879    6879      NaN        151


# Fill the missing values

In [7]:
# Fill missing CityId in User table with 0 (matches the existing "unknown" placeholder pattern)
user['CityId'] = user['CityId'].fillna(0)

# Convert CityId back to integer (it became float because of the NaNs; now that they're filled, we can convert)
user['CityId'] = user['CityId'].astype(int)

# Fill missing CityName in City table with "-" (matches existing placeholder pattern, e.g. CityId 0)
city['CityName'] = city['CityName'].fillna('-')

# Verify both fixes
print("Missing values in User now:", user.isnull().sum().sum())
print("Missing values in City now:", city.isnull().sum().sum())
print("\nUser.CityId dtype:", user['CityId'].dtype)

Missing values in User now: 0
Missing values in City now: 0

User.CityId dtype: int64


# Save the cleaned data

In [8]:
import os

# Create a folder for cleaned data (keeps raw and cleaned separate — good practice)
os.makedirs("../data/cleaned", exist_ok=True)

# Save all cleaned tables
transaction = pd.read_excel("../data/Transaction.xlsx")  # unchanged, but saving for consistency
continent = pd.read_excel("../data/Continent.xlsx")       # unchanged
country = pd.read_excel("../data/Country.xlsx")           # unchanged
region = pd.read_excel("../data/Region.xlsx")              # unchanged
attraction_type = pd.read_excel("../data/Type.xlsx")       # unchanged
visit_mode = pd.read_excel("../data/Mode.xlsx")            # unchanged

user.to_excel("../data/cleaned/User_cleaned.xlsx", index=False)
city.to_excel("../data/cleaned/City_cleaned.xlsx", index=False)
item.to_excel("../data/cleaned/Item_cleaned.xlsx", index=False)
transaction.to_excel("../data/cleaned/Transaction_cleaned.xlsx", index=False)
continent.to_excel("../data/cleaned/Continent_cleaned.xlsx", index=False)
country.to_excel("../data/cleaned/Country_cleaned.xlsx", index=False)
region.to_excel("../data/cleaned/Region_cleaned.xlsx", index=False)
attraction_type.to_excel("../data/cleaned/Type_cleaned.xlsx", index=False)
visit_mode.to_excel("../data/cleaned/Mode_cleaned.xlsx", index=False)

print("All cleaned tables saved to data/cleaned/")

All cleaned tables saved to data/cleaned/
